# Phase 1 — Reason Taxonomy

Assigns each retraction to one attribution category from its Retraction Watch
reason labels.

**Input:** `data/raw/retraction_watch.csv`

**Outputs:**

| File | Contents |
|---|---|
| `data/interim/phase01_classified.csv` | one row per retraction, with its category |
| `data/interim/phase01_taxonomy_review.csv` | one row per label: text, frequency, assigned category |

Retraction Watch supplies a flat vocabulary of 112 labels with no groupings.
The categories below are a construction of this study, built on an attribution
axis: who bears responsibility for the retraction. Grouping by type of failure
instead would be equally defensible and would answer a different question.

A notice carries several labels joined by semicolons, so classification is a
precedence rule across whatever a record holds rather than a mapping from
labels to categories.

In [1]:
import os
import re
import pandas as pd

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

RAW = "data/raw/retraction_watch.csv"
OUT_CLASSIFIED = "data/interim/phase01_classified.csv"
OUT_REVIEW = "data/interim/phase01_taxonomy_review.csv"

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 70)
os.makedirs("data/interim", exist_ok=True)

## The taxonomy

Categories are listed in precedence order. Where a retraction carries labels
from more than one, the first listed wins.

Precedence runs from most to least serious attribution, so a notice alleging
both fabrication and compromised peer review is classified as author
misconduct. This is conservative for the editorial-compromise category, which
the study uses as its no-attributed-fault arm.

In [2]:
CATEGORIES = [
    ("AUTHOR_MISCONDUCT", [
        r"plagiarism", r"duplication", r"falsificat", r"fabricat", r"manipulat",
        r"forged", r"fake", r"paper mill", r"ethical violation",
        r"taken from dissertation", r"euphemisms for", r"computer-aided content",
        r"computer-generated", r"misconduct by author", r"misconduct - official",
        r"breach of policy by author",
    ]),
    ("EDITORIAL_COMPROMISE", [
        r"rogue editor", r"compromised", r"peer review", r"error by journal",
    ]),
    ("HONEST_ERROR", [
        r"^error in", r"not reproducible", r"contaminat",
        r"unreliable results", r"unreliable data",
    ]),
    ("ETHICS_VIOLATION", [
        r"irb/iacuc", r"informed/patient consent", r"animal welfare",
    ]),
    ("UNCONFIRMED_CONCERNS", [
        r"^concerns/issues about", r"original data and/or images not provided",
        r"lack of approval",
    ]),
]


PROCESS_PATTERNS = [
    r"^investigation by", r"^notice", r"updated to retraction", r"upgrade/update",
    r"date of article", r"author unresponsive", r"objections by author",
    r"retract and replace", r"withdraw", r"legal reasons", r"copyright",
    r"^removed$", r"temporary removal",
]

# Five labels that were manually assigned.
LABEL_OVERRIDES = {
    "Duplication of Content through Error by Journal/Publisher": "EDITORIAL_COMPROMISE",
    "Taken via Peer Review": "AUTHOR_MISCONDUCT",
    "Hoax Paper": "AUTHOR_MISCONDUCT",
    "Salami Slicing": "AUTHOR_MISCONDUCT",
    "Concerns/Issues about Peer Review": "UNCONFIRMED_CONCERNS",
}

In [3]:
def split_reasons(s):
    if not isinstance(s, str):
        return []
    return [x.strip() for x in s.split(";") if x.strip()]


def is_process(label):
    low = label.lower()
    return any(re.search(p, low) for p in PROCESS_PATTERNS)


def category_of_label(label):
    if is_process(label):
        return "PROCESS_ONLY"
    if label in LABEL_OVERRIDES:
        return LABEL_OVERRIDES[label]
    low = label.lower()
    for cat, pats in CATEGORIES:
        if any(re.search(p, low) for p in pats):
            return cat
    return "UNCLASSIFIED"


def classify(labels):
    """One category per retraction, by precedence across its labels."""
    substantive = [l for l in labels if not is_process(l)]
    if not substantive:
        return "PROCESS_ONLY"
    assigned = [category_of_label(l) for l in substantive]
    for cat, _ in CATEGORIES:
        if cat in assigned:
            return cat
    return "UNCLASSIFIED"

## Load

In [4]:
df = pd.read_csv(RAW, low_memory=False)

for col in ["RetractionDate", "OriginalPaperDate"]:
    df[col] = pd.to_datetime(df[col], errors="coerce", format="mixed")
df["RetractionYear"] = df["RetractionDate"].dt.year
df["OriginalYear"] = df["OriginalPaperDate"].dt.year
df["LagYears"] = (df["RetractionDate"] - df["OriginalPaperDate"]).dt.days / 365.25

print(f"records                  {len(df):,}")
print(f"retraction date parsed   {df.RetractionDate.notna().sum():,}")
print(f"original date parsed     {df.OriginalPaperDate.notna().sum():,}")
print(f"\nRetractionNature")
print(df.RetractionNature.value_counts(dropna=False).to_string())

records                  71,799
retraction date parsed   71,535
original date parsed     71,535

RetractionNature
RetractionNature
Retraction               66287
Expression of concern     3586
Correction                1502
NaN                        264
Reinstatement              160


## Classify

In [5]:
df["ReasonLabels"] = df.Reason.apply(split_reasons)
df["NLabels"] = df.ReasonLabels.apply(len)
df["Category"] = df.ReasonLabels.apply(classify)

print("labels per retraction")
print(f"  mean    {df.NLabels.mean():.1f}")
print(f"  median  {df.NLabels.median():.0f}")
print(f"  max     {df.NLabels.max()}")
print()
counts = df.Category.value_counts()
for cat, n in counts.items():
    print(f"  {cat:<24} {n:>7,}  ({n/len(df):>5.1%})")

labels per retraction
  mean    3.9
  median  3
  max     17

  AUTHOR_MISCONDUCT         37,322  (52.0%)
  HONEST_ERROR              10,253  (14.3%)
  PROCESS_ONLY               9,195  (12.8%)
  EDITORIAL_COMPROMISE       8,194  (11.4%)
  UNCONFIRMED_CONCERNS       6,005  ( 8.4%)
  ETHICS_VIOLATION             434  ( 0.6%)
  UNCLASSIFIED                 396  ( 0.6%)


## Labels matching no category

Reported rather than dropped silently. A frequent label appearing here means
the patterns above need extending.

In [6]:
from collections import Counter

unmatched = Counter()
for labels in df.ReasonLabels:
    for l in labels:
        if category_of_label(l) == "UNCLASSIFIED":
            unmatched[l] += 1

print(f"{len(unmatched)} labels matched no category\n")
for label, n in unmatched.most_common(25):
    print(f"  {n:>7,}  {label}")
if len(unmatched) > 25:
    print(f"  {sum(n for _, n in unmatched.most_common()[25:]):>7,}  "
          f"(+{len(unmatched) - 25} further labels)")

28 labels matched no category

      917  Conflict of Interest
      838  Objections by Third Party
      345  Civil Proceedings
      283  Unreliable Image
      221  Cites Retracted Work
      142  Doing the Right Thing
      112  Complaints about Author
      106  Miscommunication with/by Author
      106  Bias Issues or Lack of Balance
       82  Publishing Ban
       78  Criminal Proceedings
       51  Misconduct by Third Party
       48  Error by Third Party
       47  Updated to Correction
       38  Not Presented at Conference
       29  Objections by Company/Institution
       26  No Further Action
       20  Miscommunication with/by Journal/Publisher
       13  Miscommunication with/by Third Party
       12  Nonpayment of Fees and/or Refusal to Pay
       12  Miscommunication with/by Company/Institution
       10  Complaints about Third Party
        9  Taken via Translation
        9  Updated to Expression of Concern
        8  Misconduct by Company/Institution
       11  (+

## Category composition

The labels carrying each category. Where one label accounts for most of a
category, that category's meaning is largely that label's meaning.

In [7]:
ARMS = ["AUTHOR_MISCONDUCT", "HONEST_ERROR", "EDITORIAL_COMPROMISE"]

for cat in ARMS:
    sub = df[df.Category == cat]
    labels = Counter()
    for row in sub.ReasonLabels:
        for l in row:
            if not is_process(l):
                labels[l] += 1
    print("-" * 74)
    print(f"{cat}   {len(sub):,} retractions, {len(labels)} distinct labels")
    print("-" * 74)
    for label, n in labels.most_common(8):
        print(f"  {n:>7,}  {label[:62]}")
    if len(labels) > 8:
        print(f"  {sum(n for _, n in labels.most_common()[8:]):>7,}  "
              f"(+{len(labels) - 8} further labels)")
    print()

--------------------------------------------------------------------------
AUTHOR_MISCONDUCT   37,322 retractions, 89 distinct labels
--------------------------------------------------------------------------
   13,299  Unreliable Results and/or Conclusions
   11,796  Paper Mill
   10,029  Concerns/Issues about Data
    9,226  Concerns/Issues about Referencing/Attributions
    9,175  Computer-Aided Content or Computer-Generated Content
    7,375  Concerns/Issues about Results and/or Conclusions
    6,847  Concerns/Issues about Peer Review
    5,551  Duplication of/in Image
   51,932  (+81 further labels)

--------------------------------------------------------------------------
HONEST_ERROR   10,253 retractions, 50 distinct labels
--------------------------------------------------------------------------
    5,578  Unreliable Results and/or Conclusions
    3,503  Concerns/Issues about Data
    2,825  Concerns/Issues about Referencing/Attributions
    2,443  Concerns/Issues about Peer 

## Where contested labels land

Peer-review labels are the ones whose placement most affects the study, since
the editorial-compromise category is used as an arm in which the listed authors
are not the source of the problem. Manipulated peer review is frequently
arranged by authors, so a notice carrying both a peer-review label and a
misconduct label must classify as misconduct.

The table below shows how often each contested label appears in each category.
Counts exceed the category sizes above because a retraction carries several
labels.

In [8]:
CONTESTED = ["Compromised Peer Review", "Rogue Editor",
             "Concerns/Issues about Peer Review",
             "Concerns/Issues about Third Party Involvement"]

rows = []
for label in CONTESTED:
    has = df.ReasonLabels.apply(lambda ls: label in ls)
    row = {"label": label}
    for cat in ARMS:
        row[cat] = int((has & (df.Category == cat)).sum())
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))

MISCONDUCT_LABELS = [
    "Paper Mill", "Falsification/Fabrication of Data",
    "Falsification/Fabrication of Image", "Misconduct by Author",
    "Manipulation of Images", "Manipulation of Data",
    "False/Forged Authorship", "False/Forged Affiliation",
    "Plagiarism of/in Article", "Plagiarism of Image",
    "Misconduct - Official Investigation(s) and/or Finding(s)",
]

edit = df[df.Category == "EDITORIAL_COMPROMISE"]
also_misconduct = edit.ReasonLabels.apply(
    lambda ls: any(m in ls for m in MISCONDUCT_LABELS))

print(f"\nEDITORIAL_COMPROMISE: {len(edit):,} retractions")
print(f"  also carrying a misconduct label: {int(also_misconduct.sum()):,} "
      f"({also_misconduct.mean():.1%})")

                                        label  AUTHOR_MISCONDUCT  HONEST_ERROR  EDITORIAL_COMPROMISE
                      Compromised Peer Review               5320             0                  6131
                                 Rogue Editor                513             0                  2719
            Concerns/Issues about Peer Review               6847          2443                  1365
Concerns/Issues about Third Party Involvement               1301            72                   371

EDITORIAL_COMPROMISE: 8,194 retractions
  also carrying a misconduct label: 0 (0.0%)


## Write

`phase01_taxonomy_review.csv` lists every label with its frequency and assigned
category, and carries an empty `override_arm` column. Filling that column and
re-running changes a label's category; precedence across a retraction's
remaining labels still decides which category wins, so an override changes a
retraction's category only where that label was the deciding one.

In [9]:
keep = ["Record ID", "Title", "Journal", "Publisher", "Subject", "Country",
        "Author", "RetractionNature", "Reason", "Category",
        "RetractionDate", "OriginalPaperDate", "RetractionYear", "OriginalYear",
        "LagYears", "NLabels", "OriginalPaperDOI", "RetractionDOI"]
keep = [c for c in keep if c in df.columns]
df[keep].to_csv(OUT_CLASSIFIED, index=False)
print(f"{OUT_CLASSIFIED}  {len(df):,} rows")

all_labels = Counter()
for labels in df.ReasonLabels:
    for l in labels:
        all_labels[l] += 1

review = pd.DataFrame([
    {"label": l, "count": n, "assigned": category_of_label(l),
     "is_process": is_process(l),
     "manual_override": LABEL_OVERRIDES.get(l, ""),
     "override_arm": ""}
    for l, n in all_labels.most_common()
])
review.to_csv(OUT_REVIEW, index=False)
print(f"{OUT_REVIEW}  {len(review):,} labels")

data/interim/phase01_classified.csv  71,799 rows
data/interim/phase01_taxonomy_review.csv  112 labels


## Summary

Three categories carry the study: author misconduct, honest error, and
editorial compromise. `ETHICS_VIOLATION`, `UNCONFIRMED_CONCERNS`,
`UNCLASSIFIED` and `PROCESS_ONLY` are excluded from the analysis, since none
implies an attribution of fault.

Phase 8 reconstructs categories from the raw `Reason` text rather than reading
this file, so a change to the taxonomy costs a Phase 8 and Phase 12 rerun and
no re-extraction.

In [10]:
in_study = df.Category.isin(ARMS)
print(f"retractions in the three arms   {in_study.sum():,}  "
      f"({in_study.mean():.1%})")
print(f"excluded                        {(~in_study).sum():,}  "
      f"({(~in_study).mean():.1%})\n")
print(df.loc[~in_study, "Category"].value_counts().to_string())

print("\nby retraction year, three arms only")
tab = (df[in_study].groupby(["RetractionYear", "Category"]).size()
         .unstack(fill_value=0))
print(tab.loc[tab.index >= 2010].to_string())

retractions in the three arms   55,769  (77.7%)
excluded                        16,030  (22.3%)

Category
PROCESS_ONLY            9195
UNCONFIRMED_CONCERNS    6005
ETHICS_VIOLATION         434
UNCLASSIFIED             396

by retraction year, three arms only
Category        AUTHOR_MISCONDUCT  EDITORIAL_COMPROMISE  HONEST_ERROR
RetractionYear                                                       
2010.0                       1588                    30           103
2011.0                       2939                    27           116
2012.0                        683                    64           153
2013.0                        742                    48           206
2014.0                        609                   121           221
2015.0                        846                   202           288
2016.0                        919                   213           332
2017.0                        964                   244           303
2018.0                       1173        